# Team-Conditioned World Cup DDColor Fine-Tuning

The notebook fine-tunes the CLIP team-conditioned DDColor tiny architecture for World Cup image colorization on a two-GPU CUDA-backed remote Jupyter runtime.

The public DDColorWorldCup repository is cloned inside the session when it is not already available, dependencies are installed inside the notebook, image/team rows are read from `/kaggle/input/worldcup_team_metadata.csv`, and outputs are written under `/kaggle/working`.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/rubiotelmo/DDColorWorldCup.git"
REPO_ROOT = Path("/kaggle/working/DDColorWorldCup") if Path("/kaggle").exists() else Path.cwd()
if not (REPO_ROOT / "ddcolor").exists():
    REPO_ROOT = Path.cwd() / "DDColorWorldCup"
    if not (REPO_ROOT / "ddcolor").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
REPO_ROOT

PosixPath('/kaggle/working/DDColorWorldCup')

In [2]:
# Kaggle already includes a CUDA PyTorch stack; reinstalling torch can break NCCL package metadata.
%pip install -q opencv-python-headless scikit-image tqdm huggingface-hub matplotlib transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import csv
import os
import random
import subprocess

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from huggingface_hub import PyTorchModelHubMixin
from skimage import color
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import DataLoader, Dataset
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

from basicsr.archs.discriminator_arch import DynamicUNetDiscriminator
from basicsr.losses.losses import ColorfulnessLoss, GANLoss, PerceptualLoss
from basicsr.metrics.colorfulness import calculate_cf
from basicsr.utils.img_util import tensor_lab2rgb
from ddcolor import DDColor

assert (REPO_ROOT / "wcddcolor" / "model.py").exists(), "wcddcolor/model.py is missing. Push or upload the updated repo before running this notebook."
from wcddcolor.model import WorldCupDDColor

DATA_ROOT = Path("/kaggle/input")
TEAM_CSV = DATA_ROOT / "worldcup-images-2018-2022/metadata.csv"
WORK_DIR = Path("/kaggle/working/wcddcolor_team_runs")
IMG_SIZE = 256
EPOCHS = 5
BATCH_SIZE = 4
LR = 1e-4
SEED = 0

assert torch.cuda.device_count() >= 2, "Two CUDA GPUs are required."
WORLD_SIZE = 2
device = torch.device("cuda:0")
WORK_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 01.- Defining the problem

World Cup image colorization is framed as Lab color prediction conditioned by match teams. The model receives the grayscale lightness channel L, represented as grayscale RGB for DDColor, and two team names whose CLIP text embeddings bias the color decoder toward plausible football kit colors.

The World Cup dataset is imported from `/kaggle/input` through a CSV file with `path`, `team1`, and `team2` columns, then split into training, validation, and testing partitions. Images without a CSV row are ignored, and CSV rows with missing team metadata are skipped.

In [ ]:
assert TEAM_CSV.exists(), f"Missing metadata CSV: {TEAM_CSV}"
with TEAM_CSV.open(newline="") as f:
    rows = list(csv.DictReader(f))

assert rows, "The team metadata CSV is empty."
assert {"path", "team1", "team2"} <= set(rows[0]), "CSV must contain path, team1, and team2 columns."
rows = [
    {"path": DATA_ROOT / row["path"], "team1": row["team1"].strip(), "team2": row["team2"].strip()}
    for row in rows
    if row["team1"].strip() and row["team2"].strip()
]
assert rows, "No rows with two team names were found."
assert all(row["path"].exists() for row in rows), "Every labeled row needs an existing image path."
random.Random(SEED).shuffle(rows)

n_test = max(1, int(0.1 * len(rows)))
n_val = max(1, int(0.1 * len(rows)))
test_rows = rows[:n_test]
val_rows = rows[n_test:n_test + n_val]
train_rows = rows[n_test + n_val:]

assert train_rows and val_rows and test_rows, "At least three labeled metadata rows are required."
len(train_rows), len(val_rows), len(test_rows)

In [ ]:
sample_rows = train_rows[:4]
fig, axes = plt.subplots(1, len(sample_rows), figsize=(3 * len(sample_rows), 3))
if len(sample_rows) == 1:
    axes = [axes]
for ax, row in zip(axes, sample_rows):
    img = cv2.cvtColor(cv2.imread(str(row["path"])), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f'{row["team1"]} vs {row["team2"]}', fontsize=9)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 02.- Choose a measure of success

The training objective follows the DDColor work with AB L1 reconstruction, VGG perceptual similarity, adversarial realism, and colorfulness regularization.

## 03.- Deciding on an evaluation protocol

The evaluation protocol uses PSNR, SSIM, and predicted-image colorfulness on validation and test images, so model selection does not depend on training loss.

## 04.- Preparing the data

Images are resized, randomly flipped during training, decolored through the Lab lightness channel, and separated into L and AB tensors. Each item also returns the two team names used by the team-conditioned color decoder.

In [ ]:
class WorldCupTeamLabDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows = rows
        self.train = train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = cv2.imread(str(row["path"]), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        if self.train and random.random() < 0.5:
            img = np.ascontiguousarray(img[:, ::-1])
        lab = color.rgb2lab(img.astype(np.float32) / 255.0).astype(np.float32)
        l = torch.from_numpy(lab[:, :, :1].transpose(2, 0, 1))
        ab = torch.from_numpy(lab[:, :, 1:].transpose(2, 0, 1))
        return l, ab, row["team1"], row["team2"]


train_loader = DataLoader(WorldCupTeamLabDataset(train_rows, True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
train_eval_rows = train_rows[:1024]
val_eval_rows = val_rows[:2048]
train_eval_loader = DataLoader(WorldCupTeamLabDataset(train_eval_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(WorldCupTeamLabDataset(val_eval_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(WorldCupTeamLabDataset(test_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
def l_to_rgb(l):
    return tensor_lab2rgb(torch.cat([l, torch.zeros_like(l), torch.zeros_like(l)], dim=1))


def lab_to_rgb(l, ab):
    return tensor_lab2rgb(torch.cat([l, ab], dim=1))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    psnr, ssim, cf = [], [], []
    for l, ab, team1, team2 in loader:
        l, ab = l.to(device), ab.to(device)
        teams = list(zip(team1, team2))
        pred_ab = model(l_to_rgb(l), teams=teams)
        pred_rgb = lab_to_rgb(l, pred_ab).cpu().clamp(0, 1)
        gt_rgb = lab_to_rgb(l, ab).cpu().clamp(0, 1)
        for pred, gt in zip(pred_rgb, gt_rgb):
            pred = pred.permute(1, 2, 0).numpy()
            gt = gt.permute(1, 2, 0).numpy()
            psnr.append(peak_signal_noise_ratio(gt, pred, data_range=1.0))
            ssim.append(structural_similarity(gt, pred, channel_axis=2, data_range=1.0))
            cf.append(calculate_cf((pred[:, :, ::-1] * 255).round().astype(np.uint8)))
    return {"psnr": float(np.mean(psnr)), "ssim": float(np.mean(ssim)), "colorfulness": float(np.mean(cf))}

## 05.- Fine tune the team-conditioned DDColor tiny model

The DDColor tiny checkpoint is loaded from Hugging Face and copied into `WorldCupDDColor` with `strict=False`, leaving the CLIP projection and team cross-attention layers freshly initialized. Fine-tuning is launched with torchrun and DistributedDataParallel: two Python processes are started, one per GPU, each process trains on a dataset shard through DistributedSampler, gradients are synchronized by NCCL after backward passes, and only rank 0 saves checkpoints and evaluates metric subsets.

In [ ]:
class DDColorHF(DDColor, PyTorchModelHubMixin):
    def __init__(self, config=None, **kwargs):
        if isinstance(config, dict):
            kwargs = {**config, **kwargs}
        super().__init__(**kwargs)


base_model = DDColorHF.from_pretrained("piddnad/ddcolor_paper_tiny")
model = WorldCupDDColor(
    encoder_name="convnext-t",
    input_size=(IMG_SIZE, IMG_SIZE),
    num_output_channels=2,
    num_queries=100,
    last_norm="Spectral",
    do_normalize=False,
).to(device)
model.load_state_dict(base_model.state_dict(), strict=False)
for p in model.encoder.parameters():
    p.requires_grad = False

l, ab, team1, team2 = next(iter(train_loader))
with torch.no_grad():
    y = model(l_to_rgb(l[:1].to(device)), teams=list(zip(team1[:1], team2[:1])))
assert y.shape == (1, 2, IMG_SIZE, IMG_SIZE)
del model, base_model, l, ab, y
torch.cuda.empty_cache()

In [ ]:
train_script = r'''
import csv
import os
import random
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.distributed as dist
from huggingface_hub import PyTorchModelHubMixin
from skimage import color
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))

from basicsr.archs.discriminator_arch import DynamicUNetDiscriminator
from basicsr.losses.losses import ColorfulnessLoss, GANLoss, PerceptualLoss
from basicsr.metrics.colorfulness import calculate_cf
from basicsr.utils.img_util import tensor_lab2rgb
from ddcolor import DDColor
from wcddcolor.model import WorldCupDDColor

DATA_ROOT = Path('/kaggle/input')
TEAM_CSV = Path(os.environ['TEAM_CSV'])
WORK_DIR = Path('/kaggle/working/wcddcolor_team_runs')
IMG_SIZE = int(os.environ['IMG_SIZE'])
EPOCHS = int(os.environ['EPOCHS'])
BATCH_SIZE = int(os.environ['BATCH_SIZE'])
LR = float(os.environ['LR'])
SEED = int(os.environ['SEED'])


class DDColorHF(DDColor, PyTorchModelHubMixin):
    def __init__(self, config=None, **kwargs):
        if isinstance(config, dict):
            kwargs = {**config, **kwargs}
        super().__init__(**kwargs)


class WorldCupTeamLabDataset(Dataset):
    def __init__(self, rows, train=False):
        self.rows = rows
        self.train = train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        img = cv2.imread(str(row['path']), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        if self.train and random.random() < 0.5:
            img = np.ascontiguousarray(img[:, ::-1])
        lab = color.rgb2lab(img.astype(np.float32) / 255.0).astype(np.float32)
        l = torch.from_numpy(lab[:, :, :1].transpose(2, 0, 1))
        ab = torch.from_numpy(lab[:, :, 1:].transpose(2, 0, 1))
        return l, ab, row['team1'], row['team2']


def l_to_rgb(l):
    return tensor_lab2rgb(torch.cat([l, torch.zeros_like(l), torch.zeros_like(l)], dim=1))


def lab_to_rgb(l, ab):
    return tensor_lab2rgb(torch.cat([l, ab], dim=1))


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    psnr, ssim, cf = [], [], []
    for l, ab, team1, team2 in loader:
        l, ab = l.to(device), ab.to(device)
        teams = list(zip(team1, team2))
        pred_ab = model(l_to_rgb(l), teams=teams)
        pred_rgb = lab_to_rgb(l, pred_ab).cpu().clamp(0, 1)
        gt_rgb = lab_to_rgb(l, ab).cpu().clamp(0, 1)
        for pred, gt in zip(pred_rgb, gt_rgb):
            pred = pred.permute(1, 2, 0).numpy()
            gt = gt.permute(1, 2, 0).numpy()
            psnr.append(peak_signal_noise_ratio(gt, pred, data_range=1.0))
            ssim.append(structural_similarity(gt, pred, channel_axis=2, data_range=1.0))
            cf.append(calculate_cf((pred[:, :, ::-1] * 255).round().astype(np.uint8)))
    return {'psnr': float(np.mean(psnr)), 'ssim': float(np.mean(ssim)), 'colorfulness': float(np.mean(cf))}


def split_rows():
    with TEAM_CSV.open(newline='') as f:
        rows = list(csv.DictReader(f))
    assert rows, 'The team metadata CSV is empty.'
    rows = [
        {'path': DATA_ROOT / row['path'], 'team1': row['team1'].strip(), 'team2': row['team2'].strip()}
        for row in rows
        if row['team1'].strip() and row['team2'].strip()
    ]
    assert rows, 'No rows with two team names were found.'
    assert all(row['path'].exists() for row in rows)
    random.Random(SEED).shuffle(rows)
    n_test = max(1, int(0.1 * len(rows)))
    n_val = max(1, int(0.1 * len(rows)))
    return rows[n_test + n_val:], rows[n_test:n_test + n_val]


def main():
    local_rank = int(os.environ['LOCAL_RANK'])
    rank = int(os.environ['RANK'])
    world_size = int(os.environ['WORLD_SIZE'])
    torch.cuda.set_device(local_rank)
    device = torch.device(f'cuda:{local_rank}')
    dist.init_process_group(backend='nccl')
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if rank == 0:
        WORK_DIR.mkdir(parents=True, exist_ok=True)
    train_rows, val_rows = split_rows()
    train_dataset = WorldCupTeamLabDataset(train_rows, True)
    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=SEED)
    per_gpu_batch_size = max(1, BATCH_SIZE // world_size)
    train_loader = DataLoader(train_dataset, batch_size=per_gpu_batch_size, sampler=train_sampler, num_workers=2, pin_memory=True)
    if rank == 0:
        train_eval_rows = train_rows[:1024]
        val_eval_rows = val_rows[:2048]
        train_eval_loader = DataLoader(WorldCupTeamLabDataset(train_eval_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        val_loader = DataLoader(WorldCupTeamLabDataset(val_eval_rows), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    base_model = DDColorHF.from_pretrained('piddnad/ddcolor_paper_tiny')
    base_model = base_model.to(device)
    model = WorldCupDDColor(
        encoder_name='convnext-t', input_size=(IMG_SIZE, IMG_SIZE), num_output_channels=2,
        num_queries=100, last_norm='Spectral', do_normalize=False
    ).to(device)
    model.load_state_dict(base_model.state_dict(), strict=False)
    del base_model
    for p in model.encoder.parameters():
        p.requires_grad = False
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)
    disc = DDP(DynamicUNetDiscriminator(n_channels=3, nf=64).to(device), device_ids=[local_rank], output_device=local_rank)

    pixel_loss = torch.nn.L1Loss()
    perceptual_loss = PerceptualLoss(
        layer_weights={'conv1_1': 0.0625, 'conv2_1': 0.125, 'conv3_1': 0.25, 'conv4_1': 0.5, 'conv5_1': 1.0},
        vgg_type='vgg16_bn', use_input_norm=True, range_norm=False, perceptual_weight=5.0, style_weight=0, criterion='l1'
    ).to(device)
    gan_loss = GANLoss('vanilla', loss_weight=1.0).to(device)
    color_loss = ColorfulnessLoss(loss_weight=0.5).to(device)
    opt_g = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=0.01, betas=(0.9, 0.99))
    opt_d = torch.optim.Adam(disc.parameters(), lr=LR, betas=(0.9, 0.99))

    start_epoch = 0
    best_ssim = -1.0
    history = []
    checkpoint_path = WORK_DIR / 'checkpoint_team.pt'
    best_path = WORK_DIR / 'best_wcddcolor_team_tiny.pt'
    if checkpoint_path.exists():
        state = torch.load(checkpoint_path, map_location=device)
        model.module.load_state_dict(state['model'], strict=False)
        disc.module.load_state_dict(state['discriminator'])
        opt_g.load_state_dict(state['opt_g'])
        opt_d.load_state_dict(state['opt_d'])
        start_epoch = state['epoch'] + 1
        best_ssim = state['best_ssim']
        history = state.get('history', [])

    for epoch in range(start_epoch, EPOCHS):
        train_sampler.set_epoch(epoch)
        model.train()
        model.module.encoder.eval()
        disc.train()
        running_g, running_d = 0.0, 0.0
        iterator = train_loader if rank else tqdm(train_loader, desc=f'epoch {epoch + 1}/{EPOCHS}')
        for l, ab, team1, team2 in iterator:
            l, ab = l.to(device), ab.to(device)
            teams = list(zip(team1, team2))
            gray_rgb = l_to_rgb(l)
            gt_rgb = lab_to_rgb(l, ab)

            for p in disc.parameters():
                p.requires_grad_(False)
            opt_g.zero_grad(set_to_none=True)
            pred_ab = model(gray_rgb, teams=teams)
            pred_rgb = lab_to_rgb(l, pred_ab)
            loss_perc, _ = perceptual_loss(pred_rgb, gt_rgb)
            loss_g = 0.1 * pixel_loss(pred_ab, ab) + loss_perc + gan_loss(disc.module(pred_rgb), True, is_disc=False) + color_loss(pred_rgb)
            loss_g.backward()
            opt_g.step()

            for p in disc.parameters():
                p.requires_grad_(True)
            opt_d.zero_grad(set_to_none=True)
            loss_d = gan_loss(disc(gt_rgb), True, is_disc=True) + gan_loss(disc(pred_rgb.detach()), False, is_disc=True)
            loss_d.backward()
            opt_d.step()
            running_g += loss_g.item()
            running_d += loss_d.item()

        totals = torch.tensor([running_g, running_d, len(train_loader)], device=device)
        dist.all_reduce(totals, op=dist.ReduceOp.SUM)
        dist.barrier()
        if rank == 0:
            train_metrics = evaluate(model.module, train_eval_loader, device)
            val_metrics = evaluate(model.module, val_loader, device)
            epoch_history = {
                'epoch': epoch + 1,
                'g_loss': (totals[0] / totals[2]).item(),
                'd_loss': (totals[1] / totals[2]).item(),
                **{f'train_{k}': v for k, v in train_metrics.items()},
                **{f'val_{k}': v for k, v in val_metrics.items()},
            }
            history.append(epoch_history)
            if val_metrics['ssim'] > best_ssim:
                best_ssim = val_metrics['ssim']
                torch.save({'model': model.module.state_dict(), 'epoch': epoch, 'metrics': val_metrics}, best_path)
            torch.save({
                'model': model.module.state_dict(), 'discriminator': disc.module.state_dict(),
                'opt_g': opt_g.state_dict(), 'opt_d': opt_d.state_dict(), 'epoch': epoch,
                'best_ssim': best_ssim, 'metrics': val_metrics, 'history': history,
            }, checkpoint_path)
            print(epoch_history)
        dist.barrier()
    dist.destroy_process_group()


if __name__ == '__main__':
    main()

'''

(WORK_DIR / 'ddp_train_worldcup_team.py').write_text(train_script)

In [ ]:
env = os.environ.copy()
env.update({
    "IMG_SIZE": str(IMG_SIZE), "EPOCHS": str(EPOCHS), "BATCH_SIZE": str(BATCH_SIZE),
    "LR": str(LR), "SEED": str(SEED), "TEAM_CSV": str(TEAM_CSV)
})
subprocess.run([
    "torchrun", "--standalone", "--nproc_per_node", str(WORLD_SIZE), str(WORK_DIR / "ddp_train_worldcup_team.py")
], check=True, env=env)

checkpoint_path = WORK_DIR / "checkpoint_team.pt"
best_path = WORK_DIR / "best_wcddcolor_team_tiny.pt"
history = torch.load(checkpoint_path, map_location="cpu").get("history", [])

Learning curves compare training and validation metrics across epochs so overfitting can be inspected after fine-tuning.

In [ ]:
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["psnr", "ssim", "colorfulness"]):
    ax.plot(epochs, [row[f"train_{metric}"] for row in history], label="train")
    ax.plot(epochs, [row[f"val_{metric}"] for row in history], label="validation")
    ax.set_title(metric)
    ax.set_xlabel("epoch")
    ax.grid(True)
    ax.legend()
fig.tight_layout()
fig.savefig(WORK_DIR / "learning_curves_team.png", dpi=150)
plt.show()

## 06.- Final model testing

The best validation checkpoint is evaluated on the held-out test partition, and a compact visual grid records grayscale inputs, team-conditioned predictions, and ground truth images.

In [ ]:
model = WorldCupDDColor(
    encoder_name="convnext-t",
    input_size=(IMG_SIZE, IMG_SIZE),
    num_output_channels=2,
    num_queries=100,
    last_norm="Spectral",
    do_normalize=False,
).to(device)
best = torch.load(best_path, map_location=device)
model.load_state_dict(best["model"], strict=False)
test_metrics = evaluate(model, test_loader)
print(test_metrics)

l, ab, team1, team2 = next(iter(test_loader))
l, ab = l[:4].to(device), ab[:4].to(device)
teams = list(zip(team1[:4], team2[:4]))
model.eval()
with torch.no_grad():
    gray_rgb = l_to_rgb(l).cpu().clamp(0, 1)
    pred_rgb = lab_to_rgb(l, model(l_to_rgb(l), teams=teams)).cpu().clamp(0, 1)
    gt_rgb = lab_to_rgb(l, ab).cpu().clamp(0, 1)

grid = make_grid(torch.cat([gray_rgb, pred_rgb, gt_rgb], dim=0), nrow=l.size(0))
save_image(grid, WORK_DIR / "test_examples_team.png")